In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [2]:
try:
    client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please ensure your OPENAI_API_KEY is set in your environment variables.")

In [3]:
def lookup_domain_info(domain: str) -> str:
    print(f"-> TOOL ACTIVATED: Looking up domain info for {domain}...")
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }
    info = mock_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})
    return json.dumps(info)

In [6]:
def check_crm_history(email: str) -> str:
    print(f"-> TOOL ACTIVATED: Checking CRM history for {email}...")
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }
    history = mock_data.get(email, mock_data["default"])
    return json.dumps(history)

In [4]:
def calculate_lead_score(data_summary: str) -> str:
    print("-> TOOL ACTIVATED: Calculating lead score...")
    data = json.loads(data_summary)
    score = "Low" # Default score
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"
    return json.dumps({"lead_score": score})

In [7]:
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info": lookup_domain_info,
    "check_crm_history": check_crm_history,
    "calculate_lead_score": calculate_lead_score
}

In [8]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info",
            "description": "Retrieves general business information (industry, size, revenue) about a company based on its domain name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {"type": "string", "description": "The company's domain name, e.g., 'acmecorp.com'"},
                },
                "required": ["domain"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history",
            "description": "Checks the internal CRM system for past contact, status, and notes associated with a specific lead email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The full email address of the lead."},
                },
                "required": ["email"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score",
            "description": "Calculates the priority score (High/Medium/Low) for a lead based on a summary of all collected domain and CRM history data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {"type": "string", "description": "A JSON string containing the combined domain_info and crm_history."},
                },
                "required": ["data_summary"],
            },
        },
    },
]

In [9]:
def run_agent(user_prompt: str):
    print(f"\n--- Running Lead Qualifier Agent ---")

    system_prompt = (
        "You are an expert CRM Lead Qualifier Agent. Your sole task is to analyze a sales lead "
        "provided via email address. You must follow these steps precisely: "
        "1. Identify the domain from the email. "
        "2. Call `lookup_domain_info` and `check_crm_history` sequentially to gather all data. "
        "3. Combine all collected data into a single JSON object. "
        "4. Call `calculate_lead_score` with the combined JSON object. "
        "5. Finally, synthesize all information (domain info, CRM history, and score) "
        "into a single, easy-to-read summary for a busy sales rep."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    collected_data = {}

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto",
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            tool_calls = response_message.tool_calls

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                if not function_to_call:
                    print(f"Error: Unknown function {function_name}")
                    continue

                function_result = function_to_call(**function_args)

                if function_name == "lookup_domain_info":
                    collected_data["domain_info"] = json.loads(function_result)
                elif function_name == "check_crm_history":
                    collected_data["crm_history"] = json.loads(function_result)
                elif function_name == "calculate_lead_score":
                    # Inject the accumulated data from previous turns
                    function_args = {"data_summary": json.dumps(collected_data)}
                    function_result = function_to_call(**function_args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_result
                })

        else:
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message.content)
            break

In [10]:
lead_email_1 = "jane@acmecorp.com"
run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")


--- Running Lead Qualifier Agent ---

[AI Thinking...]
-> TOOL ACTIVATED: Looking up domain info for acmecorp.com...
-> TOOL ACTIVATED: Checking CRM history for jane@acmecorp.com...

[AI Thinking...]
-> TOOL ACTIVATED: Calculating lead score...
-> TOOL ACTIVATED: Calculating lead score...

[AI Thinking...]

--- FINAL AGENT SUMMARY ---
Here's a summary of the lead qualification for Jane at ACMECorp:

**Domain Information:**
- **Industry:** Software/SaaS
- **Company Size:** 501-1000 employees
- **Revenue:** $50M - $100M

**CRM History:**
- **Last Contact:** November 15, 2025
- **Status:** Cold Lead
- **Notes:** Jane attended a webinar, but there has been no follow-up yet.

**Lead Score:** Medium

This suggests Jane might be a potential lead with room for engagement, especially given the company's significant size and revenue in the dynamic Software/SaaS industry. It might be beneficial to reach out and follow up on the webinar attendance to warm up this lead.




In [11]:
lead_email_2 = "bob@widgetco.net"
run_agent(f"Can you run an analysis on this lead: {lead_email_2}")


--- Running Lead Qualifier Agent ---

[AI Thinking...]
-> TOOL ACTIVATED: Looking up domain info for widgetco.net...
-> TOOL ACTIVATED: Checking CRM history for bob@widgetco.net...

[AI Thinking...]
-> TOOL ACTIVATED: Calculating lead score...
-> TOOL ACTIVATED: Calculating lead score...

[AI Thinking...]

--- FINAL AGENT SUMMARY ---
Here's the summary for the lead with email address bob@widgetco.net:

- **Domain Information:**
  - **Industry:** Manufacturing
  - **Company Size:** 100-250 employees
  - **Annual Revenue:** $10M - $25M

- **CRM History:**
  - **Last Contact Date:** 2025-12-01
  - **Status:** Active Opportunity
  - **Notes:** Discussed Q1 budget and product integration.

- **Lead Score:** High

This lead represents a significant opportunity, attributed to an active status and recent engagement around budget and product discussions. Ideal for priority follow-up by sales.
